In [1]:
import pandas as pd
import plotly.express as px


In [2]:
df = pd.read_excel("data/get_around_delay_analysis.xlsx")
df.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [3]:
print(df.shape)
df.info()

(21310, 7)
<class 'pandas.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   rental_id                                   21310 non-null  int64  
 1   car_id                                      21310 non-null  int64  
 2   checkin_type                                21310 non-null  str    
 3   state                                       21310 non-null  str    
 4   delay_at_checkout_in_minutes                16346 non-null  float64
 5   previous_ended_rental_id                    1841 non-null   float64
 6   time_delta_with_previous_rental_in_minutes  1841 non-null   float64
dtypes: float64(3), int64(2), str(2)
memory usage: 1.4 MB


In [4]:
df["rental_id"].is_unique

True

In [5]:
df.isna().sum()

rental_id                                         0
car_id                                            0
checkin_type                                      0
state                                             0
delay_at_checkout_in_minutes                   4964
previous_ended_rental_id                      19469
time_delta_with_previous_rental_in_minutes    19469
dtype: int64

In [6]:
df["delay_at_checkout_in_minutes"].describe()

count    16346.000000
mean        59.701517
std       1002.561635
min     -22433.000000
25%        -36.000000
50%          9.000000
75%         67.000000
max      71084.000000
Name: delay_at_checkout_in_minutes, dtype: float64

In [18]:
df['delay_at_checkout_in_minutes'].notna().groupby(df['state']).value_counts()

state     delay_at_checkout_in_minutes
canceled  False                            3264
          True                                1
ended     True                            16345
          False                            1700
Name: count, dtype: int64

In [7]:
fig = px.histogram(df, x="delay_at_checkout_in_minutes", nbins=100)
fig.show()

In [8]:
bornes = [-float("inf"), 0] + list(range(30, 301, 30)) + [float("inf")]
labels = ["A l'heure ou en avance"] + [f"{a+1}-{a+30} min" for a in range(0, 300, 30)] + ["+ de 300 min"]

plage_retard = pd.cut(df["delay_at_checkout_in_minutes"], bins=bornes, labels=labels).rename("plage_retard")

fig = px.histogram(plage_retard, x="plage_retard", category_orders={"plage_retard": labels})
fig.show()

In [9]:
df['delay_at_checkout_in_minutes'].isna().groupby(df['state']).sum()
#on ne peut rien en déduire pour les manquantes

state
canceled    3264
ended       1700
Name: delay_at_checkout_in_minutes, dtype: int64

In [10]:
(df["previous_ended_rental_id"].isna() == df["time_delta_with_previous_rental_in_minutes"].isna()).sum()

np.int64(21310)

In [11]:
rentals_with_previous = df[df['previous_ended_rental_id'].notna()].copy()
rentals_with_previous['previous_ended_rental_id'] = rentals_with_previous['previous_ended_rental_id'].astype(int)

rentals = rentals_with_previous.merge(df, how='left', left_on='previous_ended_rental_id', right_on='rental_id', suffixes=('', '_previous'))
print(rentals.shape)
rentals.head()

(1841, 14)


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,rental_id_previous,car_id_previous,checkin_type_previous,state_previous,delay_at_checkout_in_minutes_previous,previous_ended_rental_id_previous,time_delta_with_previous_rental_in_minutes_previous
0,511639,370585,connect,ended,-15.0,563782,570.0,563782,370585,connect,ended,136.0,NaN,NaN
1,519491,312389,mobile,ended,58.0,545639,420.0,545639,312389,mobile,ended,140.0,NaN,NaN
2,521156,392479,mobile,ended,NaN,537298,0.0,537298,392479,mobile,ended,NaN,NaN,NaN
3,525044,349751,mobile,ended,NaN,510607,60.0,510607,349751,mobile,ended,-113.0,NaN,NaN
4,528808,181625,connect,ended,-76.0,557404,330.0,557404,181625,connect,ended,-352.0,554077.0,120.0


In [12]:
rentals = rentals.drop(columns=[
    'delay_at_checkout_in_minutes',
    'previous_ended_rental_id_previous',
    'time_delta_with_previous_rental_in_minutes_previous',
    'rental_id_previous',
    'car_id_previous',
])
print(rentals.shape)
rentals.head()

(1841, 9)


,rental_id,car_id,checkin_type,state,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,checkin_type_previous,state_previous,delay_at_checkout_in_minutes_previous
0,511639,370585,connect,ended,563782,570.0,connect,ended,136.0
1,519491,312389,mobile,ended,545639,420.0,mobile,ended,140.0
2,521156,392479,mobile,ended,537298,0.0,mobile,ended,NaN
3,525044,349751,mobile,ended,510607,60.0,mobile,ended,-113.0
4,528808,181625,connect,ended,557404,330.0,connect,ended,-352.0


In [13]:
rentals.info()

<class 'pandas.DataFrame'>
RangeIndex: 1841 entries, 0 to 1840
Data columns (total 9 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   rental_id                                   1841 non-null   int64  
 1   car_id                                      1841 non-null   int64  
 2   checkin_type                                1841 non-null   str    
 3   state                                       1841 non-null   str    
 4   previous_ended_rental_id                    1841 non-null   int64  
 5   time_delta_with_previous_rental_in_minutes  1841 non-null   float64
 6   checkin_type_previous                       1841 non-null   str    
 7   state_previous                              1841 non-null   str    
 8   delay_at_checkout_in_minutes_previous       1729 non-null   float64
dtypes: float64(2), int64(3), str(4)
memory usage: 172.3 KB


In [19]:
rentals['state'].value_counts()

state
ended       1612
canceled     229
Name: count, dtype: int64

In [21]:
rentals['delay_at_checkout_in_minutes_previous'].notna().sum()

np.int64(1729)

In [24]:
rentals['delay_at_checkout_in_minutes_previous'].notna().groupby(rentals['state']).value_counts

state     delay_at_checkout_in_minutes_previous
canceled  True                                      206
          False                                      23
ended     True                                     1523
          False                                      89
Name: count, dtype: int64

In [28]:
rentals_with_known_delay = rentals[rentals['delay_at_checkout_in_minutes_previous'].notna()]

en_retard = rentals_with_known_delay['delay_at_checkout_in_minutes_previous'] > 0
probleme = rentals_with_known_delay['delay_at_checkout_in_minutes_previous'] > rentals_with_known_delay['time_delta_with_previous_rental_in_minutes']

print(round(en_retard.mean()*100,2), " % part des retards sur l'ensemble")
print(probleme.sum(), "problèmes, soit ", round(probleme.mean()*100,2), " % part des retards problématiques (sur l'ensemble)")
print(round(probleme.sum() / en_retard.sum()*100,2), " % part des retards problématiques (sur les retards)")

50.49  % part des retards sur l'ensemble
218 problèmes, soit  12.61  % part des retards problématiques (sur l'ensemble)
24.97  % part des retards problématiques (sur les retards)


In [ ]:
threshold = 120

rentals_with_known_delay = rentals[rentals['delay_at_checkout_in_minutes_previous'].notna()]
rentals_with_threshold = rentals_with_known_delay[rentals_with_known_delay['time_delta_with_previous_rental_in_minutes'] >= threshold]

en_retard = rentals_with_threshold['delay_at_checkout_in_minutes_previous'] > 0
probleme = rentals_with_threshold['delay_at_checkout_in_minutes_previous'] > rentals_with_threshold['time_delta_with_previous_rental_in_minutes']

print(len(rentals_with_threshold), " locations restantes")
print(len(rentals_with_known_delay) - len(rentals_with_threshold), " locations rendues impossibles")
print(round(en_retard.mean()*100,2), " % part des retards sur l'ensemble")
print(probleme.sum(), "problèmes, soit ", round(probleme.mean()*100,2), " % part des retards problématiques (sur l'ensemble)")
print(round(probleme.sum() / en_retard.sum()*100,2), " % part des retards problématiques (sur les retards)")

1099  locations restantes
630  locations rendues impossibles
52.23  % part des retards sur l'ensemble
38 problèmes, soit  3.46  % part des retards problématiques (sur l'ensemble)
6.62  % part des retards problématiques (sur les retards)


In [33]:
rentals_with_known_delay['time_delta_with_previous_rental_in_minutes'].describe()

count    1729.000000
mean      276.454598
std       254.228780
min         0.000000
25%        60.000000
50%       180.000000
75%       540.000000
max       720.000000
Name: time_delta_with_previous_rental_in_minutes, dtype: float64

In [34]:
graph = rentals_with_known_delay.copy()
graph['statut'] = (graph['delay_at_checkout_in_minutes_previous'] > graph['time_delta_with_previous_rental_in_minutes']).map(
    {True: 'Problématique', False: 'Sans problème'}
)

fig = px.histogram(
    graph,
    x='time_delta_with_previous_rental_in_minutes',
    color='statut',
    color_discrete_map={'Sans problème': 'lightblue', 'Problématique': 'royalblue'},
    category_orders={'statut': ['Sans problème', 'Problématique']},
    title="Locations par écart avec la location précédente",
)
fig.update_traces(xbins=dict(start=-15, end=735, size=30))
fig.update_layout(bargap=0.1, xaxis_title="Écart avec la location précédente (min)", yaxis_title="Nombre de locations")
fig.show()

In [35]:
threshold = 120

graph = rentals_with_known_delay.copy()
graph['bloquee'] = graph['time_delta_with_previous_rental_in_minutes'] < threshold
graph['problematique'] = graph['delay_at_checkout_in_minutes_previous'] > graph['time_delta_with_previous_rental_in_minutes']

graph['statut'] = graph['problematique'].map({True: 'Problématique', False: 'Sans problème'}) + ' ' + graph['bloquee'].map({True: 'bloquée', False: 'non bloquée'})

fig = px.histogram(
    graph,
    x='time_delta_with_previous_rental_in_minutes',
    color='statut',
    color_discrete_map={
        'Sans problème non bloquée': 'lightblue',
        'Problématique non bloquée': 'royalblue',
        'Sans problème bloquée': '#FDBA74',
        'Problématique bloquée': '#EA580C',
    },
    category_orders={'statut': ['Sans problème non bloquée', 'Sans problème bloquée', 'Problématique non bloquée', 'Problématique bloquée']},
    title="Locations par écart avec la location précédente",
)
fig.update_traces(xbins=dict(start=-15, end=735, size=30))
fig.update_layout(bargap=0.1, xaxis_title="Écart avec la location précédente (min)", yaxis_title="Nombre de locations")
fig.show()

In [49]:
lignes = []

for threshold in range(0, 721, 30):
    rentals_with_threshold = rentals_with_known_delay[rentals_with_known_delay['time_delta_with_previous_rental_in_minutes'] >= threshold]

    en_retard = rentals_with_threshold['delay_at_checkout_in_minutes_previous'] > 0
    probleme = rentals_with_threshold['delay_at_checkout_in_minutes_previous'] > rentals_with_threshold['time_delta_with_previous_rental_in_minutes']

    lignes.append({
        'threshold': threshold,
        'locations restantes': len(rentals_with_threshold),
        'locations bloquées': len(rentals_with_known_delay) - len(rentals_with_threshold),
        'retards': en_retard.sum(),
        'problématiques': probleme.sum(),
    })

tableau_seuils = pd.DataFrame(lignes)
tableau_seuils['problématiques empêchées'] = tableau_seuils['problématiques'].iloc[0] - tableau_seuils['problématiques']
tableau_seuils

,threshold,locations restantes,locations bloquées,retards,problématiques,problématiques empêchées
0,0,1729,0,873,218,0
1,30,1468,261,757,102,116
2,60,1348,381,696,72,146
3,90,1179,550,625,46,172
4,120,1099,630,574,38,180
5,150,966,763,499,26,192
6,180,901,828,465,22,196
7,210,823,906,422,18,200
8,240,778,951,397,16,202
9,270,713,1016,365,11,207


In [56]:
gain_par_tranche = pd.DataFrame({
    'de': tableau_seuils['threshold'] - 30,
    'à': tableau_seuils['threshold'],
    'locations bloquées en plus': tableau_seuils['locations bloquées'].diff(),
    'problèmes évités en plus': tableau_seuils['problématiques empêchées'].diff(),
})

gain_par_tranche = gain_par_tranche.dropna().astype(int)
gain_par_tranche


,de,à,locations bloquées en plus,problèmes évités en plus
1,0,30,261,116
2,30,60,120,30
3,60,90,169,26
4,90,120,80,8
5,120,150,133,12
6,150,180,65,4
7,180,210,78,4
8,210,240,45,2
9,240,270,65,5
10,270,300,36,0


In [50]:
fig = px.line(
    tableau_seuils,
    x='threshold',
    y=['locations bloquées', "problématiques"],
    markers=True,
    title="Effet du délai minimum entre deux locations",
)
fig.update_layout(xaxis_title="Délai minimum (min)", yaxis_title="Nombre", legend_title_text="")
fig.show()

In [51]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=tableau_seuils['threshold'], y=tableau_seuils['locations bloquées'],
               name="Locations bloquées", mode="lines+markers", line=dict(color="darkorange")),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=tableau_seuils['threshold'], y=tableau_seuils['problématiques'],
               name="Cas problématiques", mode="lines+markers", line=dict(color="royalblue")),
    secondary_y=True,
)

fig.update_layout(title="Effet du délai minimum entre deux locations", xaxis_title="Délai minimum (min)", legend_title_text="")
fig.update_yaxes(title_text="Locations bloquées", secondary_y=False)
fig.update_yaxes(title_text="Cas problématiques restants", secondary_y=True)
fig.show()

In [52]:
en_retard = rentals_with_known_delay['delay_at_checkout_in_minutes_previous'] > 0

part_retards = en_retard.groupby(rentals_with_known_delay['checkin_type_previous']).mean() * 100
part_retards = part_retards.round(2).reset_index(name='part des retards')
part_retards

,checkin_type_previous,part des retards
0,connect,38.04
1,mobile,60.87


In [54]:
en_retard = rentals_with_known_delay['delay_at_checkout_in_minutes_previous'] > 0
probleme = rentals_with_known_delay['delay_at_checkout_in_minutes_previous'] > rentals_with_known_delay['time_delta_with_previous_rental_in_minutes']
checkin = rentals_with_known_delay['checkin_type_previous']

part_problematiques = pd.DataFrame({
    "% problématiques (sur l'ensemble)": (probleme.groupby(checkin).mean() * 100).round(2),
    "% problématiques (sur les retards)": (probleme.groupby(checkin).sum() / en_retard.groupby(checkin).sum() * 100).round(2),
}).reset_index()

part_problematiques

,checkin_type_previous,% problématiques (sur l'ensemble),% problématiques (sur les retards)
0,connect,8.78,23.08
1,mobile,15.80,25.96
